## 1. Setup and Imports

In [ ]:
# Install required packages
# These cover all dependencies needed by model.py, dataset.py, and training

# Fix protobuf version conflict first
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.3

# Install transformers and dependencies
!pip install -q transformers>=4.35.0 accelerate sentencepiece

# Install torch-geometric
!pip install -q huggingface-hub tqdm

# Note: torch, numpy, pandas, scikit-learn are pre-installed on Kaggle
print("✓ All dependencies installed")

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from pathlib import Path
import json
import numpy as np
from collections import defaultdict

# Check environment
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 2. Configuration

Update these paths to match your setup.

In [ ]:
# Paths
CHECKPOINT_DIR = './checkpoints'  # Update this to your checkpoint directory
PROJECTOR_PATH = f'{CHECKPOINT_DIR}/projector_best.pt'  # or projector_epoch_X.pt

DATA_CONFIG = {
    'test_jsonl': '../data/cora/cora_test_node_data.jsonl',
    'embedding_path': '../data/cora/multi_hop_graph_embedding.pt',
}

# Model configuration (must match training config)
MODEL_CONFIG = {
    'llama_model': 'meta-llama/Llama-3.2-3B-Instruct',
    'graph_embedding_dim': 2048,
    'projector_hidden_dim': 4096,  # Updated to match training
    'num_hops': 5,
}

EVAL_CONFIG = {
    'batch_size': 4,  # Can be larger for evaluation
    'num_workers': 2,
}

print("\n" + "="*70)
print(" "*25 + "EVALUATION CONFIG")
print("="*70)
print(f"\nCheckpoint: {PROJECTOR_PATH}")
print(f"Model: {MODEL_CONFIG['llama_model']}")
print("="*70)


## 3. Load Model and Projector Weights

In [ ]:
# Clone GitHub repo and copy model files to working directory
required_files = ['model.py', 'dataset.py', 'inference.py']

if IS_KAGGLE:
    print("="*70)
    print("Cloning GitHub repository...")
    print("="*70)
    
    # Clone your GitHub repo (UPDATE with your repo URL)
    GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
    
    # Clone repo
    !git clone {GITHUB_REPO} /kaggle/working/gwm
    
    # Copy model files from repo to working directory
    repo_path = Path("/kaggle/working/gwm/gwm_e")

    %cd {repo_path}
    !git pull
    %cd ../..
    
    if repo_path.exists():
        print(f"\n✓ Repository cloned successfully")
        print(f"Source: {repo_path}")
        
        # Copy files
        for file in required_files:
            !cp {repo_path}/{file} /kaggle/working/
            print(f"✓ Copied {file}")

    else:
        print(f"\n❌ Repository path not found: {repo_path}")
        print("Please check the repository structure")
else:
    print("Running locally - files should be in current directory")

# Verify files exist
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready: {required_files}")

In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = "HF_TOKEN"
HF_TOKEN = UserSecretsClient().get_secret(HF_TOKEN)
!hf auth login --token {HF_TOKEN}

In [ ]:
from model import GWM_E

print("="*70)
print(" "*20 + "Loading Model and Weights")
print("="*70)

# Initialize model (same as training)
print(f"\nInitializing {MODEL_CONFIG['llama_model']}...")

model = GWM_E(
    llama_model_path=MODEL_CONFIG['llama_model'],
    graph_embedding_dim=MODEL_CONFIG['graph_embedding_dim'],
    projector_hidden_dim=MODEL_CONFIG['projector_hidden_dim'],
    num_hops=MODEL_CONFIG['num_hops'],
    freeze_llm=True,
)

# Load trained projector weights
print(f"\nLoading projector weights from: {PROJECTOR_PATH}")
model.load_projector(PROJECTOR_PATH)
print("✓ Projector weights loaded successfully!")

model.eval()  # Set to evaluation mode

# Model statistics
llm_params = sum(p.numel() for p in model.llm.parameters()) / 1e9
projector_params = sum(p.numel() for p in model.projector.parameters()) / 1e6

print(f"\nModel Statistics:")
print(f"  LLaMA parameters: {llm_params:.2f}B (frozen)")
print(f"  Projector parameters: {projector_params:.2f}M (loaded)")


## 4. Load Test Dataset

In [ ]:
from dataset import create_dataloaders

print("="*70)
print(" "*20 + "Loading Test Dataset")
print("="*70)

# Create test loader (no need for train loader)
_, test_loader = create_dataloaders(
    train_jsonl=DATA_CONFIG['test_jsonl'],  # Use test for both (train not needed)
    test_jsonl=DATA_CONFIG['test_jsonl'],
    embedding_path=DATA_CONFIG['embedding_path'],
    tokenizer=model.tokenizer,
    batch_size=EVAL_CONFIG['batch_size'],
    num_workers=EVAL_CONFIG['num_workers'],
    num_hops=MODEL_CONFIG['num_hops'],
)

print(f"\n✓ Test dataset loaded successfully!")
print(f"  Test samples: {len(test_loader.dataset):,}")
print(f"  Test batches: {len(test_loader):,}")

## 5. Import Evaluation Functions

In [ ]:
# Import evaluation functions from inference.py
from inference import generate_predictions, evaluate_predictions

print("✓ Evaluation functions imported from inference.py")

## 6. Run Evaluation

In [ ]:
print("="*70)
print(" "*20 + "RUNNING EVALUATION")
print("="*70)
print()

# Use inference.py functions for evaluation
test_dataset = test_loader.dataset

predictions = generate_predictions(
    model=model,
    test_dataset=test_dataset,
    device=device,
    max_new_tokens=50,
    temperature=0.1,
)

results = evaluate_predictions(predictions)

print("\n" + "="*70)
print(" "*20 + "EVALUATION RESULTS")
print("="*70)
print(f"\n📊 Overall Metrics:")
print(f"  Test Accuracy:  {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")
print(f"  Correct:        {results['correct']:,} / {results['total']:,}")
print("\n📈 Per-Class Accuracy:")
for label, acc in sorted(results['class_accuracy'].items()):
    count = results['class_counts'][label]
    correct = int(acc * count)
    print(f"  {label:20s}: {acc:.4f} ({correct}/{count})")
print("="*70)

## 7. Sample Predictions

In [ ]:
# Detailed debugging: Show first 10 predictions with comparison
print("\n" + "="*70)
print(" "*20 + "DETAILED PREDICTION ANALYSIS")
print("="*70)

num_debug = min(10, len(predictions))
exact_matches = 0
contains_matches = 0

for i in range(num_debug):
    pred = predictions[i]
    pred_text = pred['prediction'].strip()
    gt_text = pred['ground_truth'].strip()
    
    # Check different match types
    exact_match = pred_text.lower() == gt_text.lower()
    contains_match = gt_text.lower() in pred_text.lower()
    
    if exact_match:
        exact_matches += 1
        contains_matches += 1
    elif contains_match:
        contains_matches += 1
    
    print(f"\n{'='*70}")
    print(f"Sample {i+1}:")
    print(f"  Node ID:        {pred['node_id']}")
    print(f"  Ground Truth:   '{gt_text}'")
    print(f"  Prediction:     '{pred_text}'")
    print(f"  Exact Match:    {'✓' if exact_match else '✗'}")
    print(f"  Contains Match: {'✓' if contains_match else '✗'}")

print(f"\n{'='*70}")
print(f"Debug Summary (first {num_debug} samples):")
print(f"  Exact matches:    {exact_matches}/{num_debug} ({exact_matches/num_debug*100:.1f}%)")
print(f"  Contains matches: {contains_matches}/{num_debug} ({contains_matches/num_debug*100:.1f}%)")
print(f"{'='*70}\n")


In [ ]:
# Show sample predictions
print("\n" + "="*70)
print(" "*20 + "SAMPLE PREDICTIONS")
print("="*70)

num_samples = min(5, len(predictions))
for i in range(num_samples):
    pred = predictions[i]
    match = pred['prediction'].strip().lower() == pred['ground_truth'].strip().lower()
    
    print(f"\nExample {i+1}:")
    print(f"  Node ID:      {pred['node_id']}")
    print(f"  Ground Truth: {pred['ground_truth']}")
    print(f"  Prediction:   {pred['prediction']}")
    print(f"  Correct:      {'✓' if match else '✗'}")

print("\n" + "="*70)

## 8. Save Results

In [ ]:
# Save evaluation results
eval_results = {
    'test_accuracy': results['accuracy'],
    'correct': results['correct'],
    'total': results['total'],
    'class_accuracy': results['class_accuracy'],
    'class_counts': results['class_counts'],
    'checkpoint': PROJECTOR_PATH,
    'model_config': MODEL_CONFIG,
}

output_path = f"{CHECKPOINT_DIR}/evaluation_results.json"
with open(output_path, 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"\n✓ Evaluation results saved to: {output_path}")

# Save predictions
predictions_path = f"{CHECKPOINT_DIR}/predictions.json"
with open(predictions_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

print(f"✓ Predictions saved to: {predictions_path}")

print("\n" + "="*70)
print(" "*20 + "EVALUATION COMPLETE!")
print("="*70)